In [ ]:
!pip install onnxscript
from google.colab import userdata
from huggingface_hub import HfApi
from pathlib import Path
import json
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score, precision_score, recall_score
from typing import Optional
from huggingface_hub import hf_hub_download
import sys
from copy import deepcopy

In [ ]:
def load_tokenizer_and_model_from_hf(cfg, top_dim, sub_dim, device):
    hf_repo_id = cfg.get("hf_repo")
    hf_subfolder = cfg.get("hf_encoder_subfolder")

    checkpoint_filename = "final_stage3_v4/best_stage3.pt"
    print(f"🔹 Loading Tokenizer & Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id, subfolder=hf_subfolder)
        print(f"Tokenizer loaded.")
    except Exception as e:
        print(f"Failed to load tokenizer: {e}")
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id)

    print(f" Initializing Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    model = JointModel(
            hf_repo_id,
            subfolder=hf_subfolder,
            top_dim=top_dim,
            sub_dim=sub_dim,
            sentiment_classes=cfg.get("num_classes", 2),
            aspect_emb_dim=cfg.get("aspect_embedding_dim", 768),
            dropout=0.2).to(device)

    print(f" Downloading weights from: {hf_repo_id}/{checkpoint_filename} ...")
    try:
        cached_path = hf_hub_download(
            repo_id=hf_repo_id,
            filename=checkpoint_filename)

        checkpoint = torch.load(cached_path, map_location=device)

        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint 

        missing, unexpected = model.load_state_dict(state_dict, strict=False)

        print(f" Trained weights loaded successfully!")
        if missing: print(f"   Missing keys: {len(missing)} (Ensure this matches expectation)", missing)
        if unexpected: print(f"   Unexpected keys: {len(unexpected)}", unexpected)


    except Exception as e:
        print(f"Critical Error: Could not load checkpoint from Hugging Face.")

    return tokenizer, model

In [4]:
class JointDataset(Dataset):
    def __init__(self, data, tokenizer, top_dim, sub_dim, max_len=256, label_map=None):
        self.data = data
        self.tokenizer = tokenizer
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.max_len = max_len
        self.label_map = label_map

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"]
        }

    @staticmethod
    def collate_fn(batch):
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "attention_mask": torch.stack([b["attention_mask"] for b in batch])
        }

In [5]:
class SentimentAdapter(nn.Module):
    def __init__(self, hidden_dim=768, bottleneck_dim=64):
        super().__init__()
        self.down = nn.Linear(hidden_dim, bottleneck_dim, bias=False)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck_dim, hidden_dim, bias=False)

        nn.init.zeros_(self.up.weight)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

In [ ]:
class EvalDataset(Dataset):
    def __init__(self, data, tokenizer, top_dim, sub_dim, max_len=256, label_map=None):
        self.data = data
        self.tokenizer = tokenizer
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.max_len = max_len
        self.label_map = label_map # Store label_map

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}

        top = torch.tensor(item["top_cluster_ids"], dtype=torch.float32)
        if top.ndim == 0:  # single int label
            top = F.one_hot(top.long(), num_classes=self.top_dim).float()

        sub_ids = torch.tensor(item["sub_cluster_ids"], dtype=torch.float32)

        raw_sentiments = item.get("sentiments", {})
        sentiments = {}
        for k, v in raw_sentiments.items():
            mapped_v = v
            if self.label_map:
                mapped_v = self.label_map.get(v)

            if mapped_v is not None: # Only include if a valid (non-None) value is found/mapped
                sentiments[int(k)] = mapped_v

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "top_labels": top,
            "sub_labels": sub_ids,
            "sentiments": sentiments,
        }

    @staticmethod
    def collate_fn(batch):
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
            "top_labels": torch.stack([b["top_labels"] for b in batch]),
            "sub_labels": torch.stack([b["sub_labels"] for b in batch]),
            "sentiments": [b["sentiments"] for b in batch],
        }

In [7]:
class AspectAttention(nn.Module):
    def __init__(self, hidden_dim, aspect_emb_dim, d_k=256, d_v=256):
        super().__init__()
        self.Wq = nn.Linear(aspect_emb_dim, d_k, bias=False)
        self.Wk = nn.Linear(hidden_dim, d_k, bias=False)
        self.Wv = nn.Linear(hidden_dim, d_v, bias=False)
        self.Wo = nn.Linear(d_v, hidden_dim)

    def forward(self, enc, aspect_emb, attention_mask):
        """
        enc: (B, L, H)
        aspect_emb: (B, N, E)  <-- Now expects 3D input (Standard)
        """
        # Linear layers handle the (B, N) dimensions automatically
        Q = self.Wq(aspect_emb)          # (B, N, d_k)
        K = self.Wk(enc)                 # (B, L, d_k)
        V = self.Wv(enc)                 # (B, L, d_v)

        # Compute Scores: (B, N, d_k) x (B, d_k, L) -> (B, N, L)
        scores = torch.matmul(Q, K.transpose(1, 2)).to(Q.dtype) / (Q.size(-1) ** 0.5)

        # Masking
        if attention_mask.dim() == 3:
            mask = attention_mask[:, :, 0].unsqueeze(1)
        elif attention_mask.dim() == 2:
            mask = attention_mask.unsqueeze(1)
        else:
            raise ValueError(f"Unexpected attention_mask shape: {attention_mask.shape}")
        NEG_INF = torch.tensor(-1e4, dtype=scores.dtype, device=scores.device)
        scores = scores.masked_fill(mask == 0, NEG_INF) # Safe -1e4
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V.to(attn.dtype))  # (B, N, d_v)

        return self.Wo(context)   #(B, N, H)

In [8]:
class HierarchicalClassifier(nn.Module):
    def __init__(self, encoder_name, top_dim, sub_dim, dropout=0.2, subfolder=None): # Add subfolder
        super().__init__()
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.encoder = AutoModel.from_pretrained(encoder_name, subfolder=subfolder) # Pass subfolder
        h_dim = self.encoder.config.hidden_size
        self.aspect_emb_dim = h_dim
        self.head_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.head_query = nn.Parameter(torch.empty(1, top_dim, h_dim))
        nn.init.normal_(self.head_query, mean=0, std=0.02)
        self.head_gate_proj = nn.Linear(h_dim * 2, h_dim)
        # Stage 1 (top-level)
        self.head_norm = nn.LayerNorm(h_dim)
        self.head_weight = nn.Parameter(torch.randn(top_dim, h_dim))
        self.head_bias = nn.Parameter(torch.zeros(top_dim))

        # Initialization
        torch.nn.init.normal_(self.head_weight, std=0.02)
        self.head_dropout = nn.Dropout(dropout)

        # Stage 2 (sub-level)

        self.sub_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.sub_query = nn.Parameter(torch.empty(1, sub_dim, h_dim))
        nn.init.normal_(self.sub_query, mean=0, std=0.02)
        self.sub_gate_proj = nn.Linear(h_dim * 2, h_dim)

        self.sub_norm = nn.LayerNorm(h_dim)
        self.sub_weight = nn.Parameter(torch.randn(sub_dim, h_dim))
        self.sub_bias = nn.Parameter(torch.zeros(sub_dim))

        # Initialization
        torch.nn.init.normal_(self.sub_weight, std=0.02)
        self.sub_dropout = nn.Dropout(dropout)

        # Mapping-related attributes
        self.top_to_sub_map = None          # sparse tensor (top_dim × sub_dim)

    def set_top_to_sub_map(self, mapping: dict):
        self.top_to_sub_dict = mapping  # keep for loss computation

        rows, cols = [], []
        for t, subs in mapping.items():
            rows.extend([t] * len(subs))
            cols.extend(subs)

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values = torch.ones(len(rows), dtype=torch.float32)
        sparse_map = torch.sparse_coo_tensor(indices, values, (self.top_dim, self.sub_dim))
        self.top_to_sub_map = sparse_map.coalesce()
        dense = self.top_to_sub_map.to_dense()
        np.save("top_to_sub_dense.npy", dense.numpy())


    def forward(self, input_ids, attention_mask):
        # Encoder
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        enc = outputs.last_hidden_state
        batch_size = enc.size(0)
        HALF = torch.tensor(0.5, dtype=enc.dtype, device=enc.device)
        ONE = torch.tensor(1.0, dtype=enc.dtype, device=enc.device)
        mask = attention_mask.unsqueeze(-1).float() #(B, L, 1)
        pooled = (enc * mask.to(enc.dtype)).sum(1) / mask.sum(1).clamp(min=ONE)
        top_query_expanded = self.head_query.expand(batch_size, -1, -1) # (B, N, H)
        aspect_context = self.head_aspect_attention(enc, top_query_expanded, mask) # (B, N, H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.top_dim, -1) # (B, N, H)

        # Concatenate: (B, N, 2*H)
        combined = torch.cat([pooled_expanded, aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.head_gate_proj(combined))

        # Fuse: Alpha controls how much we use Pooled vs Attention
        fused_embedding = alpha * pooled_expanded.to(alpha.dtype) + (1 - alpha) * aspect_context.to(alpha.dtype)
        fused_norm = self.head_norm(self.head_dropout(fused_embedding).float())

        # 4. Top-Level Prediction
        top_logits = (fused_norm.to(self.head_weight.dtype) * self.head_weight).sum(dim=-1) + self.head_bias
        # Now opened for sub
        p_top = torch.sigmoid(top_logits)

        # Compute sub-cluster prior
        batch_size, device, dtype = p_top.size(0), p_top.device, p_top.dtype
        sub_prior = torch.matmul(p_top, self.top_to_sub_dense.to(p_top.dtype))

        # Move thresholds to the same device as p_top
        sub_query_expanded = self.sub_query.expand(batch_size, -1, -1) # (B, N, H)
        sub_aspect_context = self.sub_aspect_attention(enc, sub_query_expanded, mask) # (B, N, H)
        # 3. Gating Logic
        # Concatenate: (B, N, 2*H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.sub_dim, -1)
        combined = torch.cat([pooled_expanded, sub_aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.sub_gate_proj(combined))

        # Fuse: Alpha controls how much we use Pooled vs Attention
        fused_embedding = alpha * pooled_expanded.to(alpha.dtype) + (1 - alpha) * sub_aspect_context.to(alpha.dtype)
        fused_norm = self.sub_norm(self.sub_dropout(fused_embedding).float())

        # 4. Top-Level Prediction
        sub_logits = (fused_norm.to(self.sub_weight.dtype) * self.sub_weight).sum(dim=-1) + self.sub_bias

        # Hierarchical gating
        sub_logits = sub_logits + HALF * sub_prior.detach()
        return top_logits, sub_logits, enc, mask



In [9]:
class JointModel(HierarchicalClassifier):
    def __init__(self, encoder_name, top_dim, sub_dim,
                 sentiment_classes=2, aspect_emb_dim=None, dropout=0.2, subfolder=None): # Add subfolder

        super().__init__(encoder_name, top_dim, sub_dim, dropout=dropout, subfolder=subfolder) # Pass subfolder
        self.sentiment_adapters = nn.ModuleList([SentimentAdapter(768, 64) for _ in range(5)])
        h_dim = self.encoder.config.hidden_size
        self.aspect_emb_dim = h_dim
        self.sent_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim, d_k=h_dim, d_v=h_dim)
        self.sent_gate_proj = nn.Linear(h_dim * 2, h_dim * 2)
        sent_in = 2*h_dim+self.aspect_emb_dim
        self.sent_norm1 = nn.LayerNorm(h_dim)
        self.sent_norm2 = nn.LayerNorm(h_dim)

        self.sent_proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(sent_in, sent_in//2),
            nn.GELU(),
            nn.Linear(sent_in//2, sent_in//4),
            nn.GELU(),
            nn.Linear(sent_in//4, 2))

    def forward(self, input_ids, attention_mask, sentiments_dict_batch=None):
        top_logits, sub_logits, enc, mask = super().forward(input_ids, attention_mask)
        ONE = torch.tensor(1.0, dtype=enc.dtype, device=enc.device)
        for adapter in self.sentiment_adapters:
            enc = adapter(enc)
        pooled = (enc * mask.to(enc.dtype)).sum(1) / mask.sum(1).clamp(min=ONE)
        B = enc.size(0)

        aspect_embs = self.sub_query.expand(B, -1, -1) # (B, N, E)

        # Returns (B, N, H)
        aspect_context = self.sent_aspect_attention(enc, aspect_embs, attention_mask)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.sub_dim, -1) # (B, N, H)

        # (B, N, H+E)
        combined = torch.cat([pooled_expanded, aspect_context], dim=-1)
        alpha = torch.sigmoid(self.sent_gate_proj(combined))
        alpha_pooled, alpha_context = alpha.chunk(2, dim=-1)

        pooled_expanded = self.sent_norm1(pooled_expanded.float())
        aspect_context = self.sent_norm2(aspect_context.float())
        fused_embedding = torch.cat([alpha_pooled * pooled_expanded.to(alpha_pooled.dtype), alpha_context * aspect_context.to(alpha_context.dtype)], dim=-1)

        feature = torch.cat([aspect_embs, fused_embedding], dim=-1)
        sent_logits = self.sent_proj(feature) #(B, N, 2)

        return top_logits, sub_logits, sent_logits



In [10]:
class JointModelInfer(nn.Module):
    def __init__(self, model: JointModel):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        top_logits, sub_logits, sent_logits = self.model(
            input_ids, attention_mask)
        return top_logits, sub_logits, sent_logits


In [11]:
def inference_aspect(self, top_logits, sub_logits, top_thresholds, sub_thresholds):
    p_top = torch.sigmoid(top_logits)        # (B, T)
    p_sub = torch.sigmoid(sub_logits)        # (B, S)
    top_thr = top_thresholds.unsqueeze(0)   # (1, T)
    sub_thr = sub_thresholds.unsqueeze(0)   # (1, S)
    p_top_bin = (p_top > top_thr).to(torch.float32)
    mask = torch.matmul(p_top_bin, self.top_to_sub_dense)  # (B, S)
    mask = mask.clamp(0.0, 1.0)
    no_top = (p_top_bin.sum(dim=1, keepdim=True) == 0).to(torch.float32)  # (B, 1)
    mask = mask + no_top * (1.0 - mask)
    p_sub_masked = p_sub * mask
    p_sub_bin = (p_sub_masked > sub_thr).to(torch.float32)
    return p_sub_bin

In [ ]:
def run(cfg, mode=None):
    DEVICE = torch.device(cfg["device"])
    sample_json = json.load(open(cfg["hierarchical_json"], "r", encoding="utf-8"))

    top_to_sub_map = defaultdict(set)
    for it in sample_json:
        for k, v in it.get("top_to_sub_ids", {}).items():
            top_to_sub_map[int(k)].update(v)
    top_to_sub_map = {k: list(v) for k, v in top_to_sub_map.items()}

    top_to_sub_map[3] = list(range(29))

    first = sample_json[0]
    top_dim = len(first["top_cluster_ids"])
    sub_dim = len(first["sub_cluster_ids"]) if "sub_cluster_ids" in first else 29 # Fallback if not directly available

    sub_thresholds = torch.tensor([0.5] * sub_dim, dtype=torch.float32, device=DEVICE)
    top_thresholds = torch.tensor([0.5] * top_dim, dtype=torch.float32, device=DEVICE)

    tokenizer, model = load_tokenizer_and_model_from_hf(
        cfg=cfg, top_dim=top_dim, sub_dim=sub_dim, device=DEVICE)

    model.set_top_to_sub_map(top_to_sub_map)
    model.eval()
    model = JointModelInfer(model)
    model = model.to(device="cuda")
    model.eval()
    if mode == 0:
        torch.set_grad_enabled(False)

        # Dataset
        dataset = JointDataset(sample_json, tokenizer, top_dim=top_dim, sub_dim=sub_dim, max_len=cfg["max_len"], label_map=cfg["label_map"])
        eval_dataset = EvalDataset(sample_json, tokenizer, top_dim=top_dim, sub_dim=sub_dim, max_len=cfg["max_len"], label_map=cfg["label_map"])

        valid_indices = np.load("/content/valid_indices (1).npy")
        all_indices = np.arange(len(dataset))
        train_indices = np.setdiff1d(all_indices, valid_indices)
        val_ds = torch.utils.data.Subset(dataset, valid_indices)
        eval_ds = torch.utils.data.Subset(eval_dataset, valid_indices)

        batch_size = 64
        #train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=JointDataset.collate_fn)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=JointDataset.collate_fn)
        eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False, collate_fn=EvalDataset.collate_fn)

        model.eval()
        sent_preds_list , pred_sub_bin_list = [], []
        for batch in tqdm(val_loader):
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            top_logits, sub_logits, sent_logits= model(ids, mask)
            pred_sub_bin = inference_aspect(model, top_logits, sub_logits, top_thresholds, sub_thresholds)
            sent_preds = (torch.argmax(sent_logits, dim=-1)) #(B, N)
            sent_preds_list.append(sent_preds)
            pred_sub_bin_list.append(pred_sub_bin)
            #sent_preds[(pred_sub_bin == 0)] = -1
        return pred_sub_bin_list, sent_preds_list, eval_loader
    else:
        return model


In [ ]:
def test(CFG):
  with torch.no_grad():
    pred_sub_bin, sent_preds, eval_loader = run(CFG, mode=0)
    pred_sub_bin = torch.cat(pred_sub_bin).cpu().numpy()
    sent_preds = torch.cat(sent_preds).cpu().numpy()

    y_sub, sentiments = [], []
    for batch in tqdm(eval_loader):
        y_sub.append(batch["sub_labels"].cpu().numpy())
        sentiments += batch["sentiments"]
    y_sub = np.concatenate(y_sub)

    # Correctly get the total number of samples for evaluation
    len_eval_data = len(eval_loader.dataset)
    sent_labels = np.full((len_eval_data, 29), -1.0)

    for i, row_i in enumerate(sentiments):
      aspect, sentiments = list(row_i.keys()), list(row_i.values())
      sent_labels[i, aspect] = sentiments

    f1_sub = f1_score(y_sub, pred_sub_bin, average="macro", zero_division=0)
    recall_sub = recall_score(y_sub, pred_sub_bin, average="macro", zero_division=0)
    precision_sub = precision_score(y_sub, pred_sub_bin, average="macro", zero_division=0)
    print("sub_pred_class_wise - precision, recall, f1 ", precision_sub, recall_sub, f1_sub)
    sent_preds_flat = sent_preds.flatten()
    sent_labels_flat = sent_labels.flatten()
    mask = sent_labels_flat != -1
    f1_sent = f1_score(sent_labels_flat[mask], sent_preds_flat[mask], average="macro", zero_division=0)

    sent_preds[(pred_sub_bin == 0)] = -1
    hard_class = macro_metrics(sent_labels, sent_preds)
    hard_sample = sample_metrics(sent_labels, sent_preds)
    print("hard_pred_class_wise", hard_class["precision"], hard_class["recall"], hard_class["f1"])
    print("hard_pred_sample_wise", hard_sample["precision"], hard_sample["recall"], hard_sample["f1"])
    sent_conf_mat = confusion_matrix(sent_labels_flat[mask], sent_preds_flat[mask])
    print("f1 sub, f1_dent", f1_sub, f1_sent)
    print(sent_conf_mat)

In [14]:
CFG = {
    "hf_repo": "Faisal191/aspect-classifier",   # repo with tokenizer/encoder or fallback to yangheng model
    "local_hf_checkpoint": None,                # optional path to stage2_compact.pt local copy (if you uploaded)
    "hf_encoder_subfolder": "Domain_trained_encoder", # Specify the subfolder separately
    "hierarchical_json": r"/content/final_aspa_data_hierarchical_with_sentiments_temp11.json",  # your prepared hierarchical JSON
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 16,
    "epochs": 10,
    "lr_encoder": 1e-5,
    "lr_heads": 5e-5,
    "lr_sent": 1e-4,
    "lr_sent_encoder": 2e-5,
    "use_amp": True,
    "weight_decay": 0.01,
    "max_len": 256,
    "freeze": True,   # True: freeze encoder (only train sentiment head); False: fine-tune encoder too
    "freeze_epochs": 10,        # Number of initial epochs to freeze the encoder
    "use_aspect_embedding": True, # alternative to one-hot: learn small embedding per sub-cluster
    "aspect_embedding_dim": 768,
    "save_dir": "./sentiment_head_ckpt",
    "seed": 42,
    # label mapping:
    # your data had sentiments as { "sub_id": 1, -1 } etc. We map them to class ids [0..C-1]
    # By default map: -1 -> 0 (neg), 0 -> 1 (neutral) if present, 1 -> 2 (pos)
    "label_map": { 0: 0, 1: 1},
    "num_classes": 2,
}
Path(CFG["save_dir"]).mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
DEVICE = torch.device(CFG["device"])

In [15]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def macro_metrics(
    y_true,
    y_pred,
    labels=(-1, 0, 1)
):
    """
    Macro class-wise metrics:
    - Compute per-aspect multiclass F1/Precision/Recall
    - Average across aspects

    y_true, y_pred: (N, S)
    """

    precisions, recalls, f1s = [], [], []

    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        precisions.append(
            precision_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        recalls.append(
            recall_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        f1s.append(
            f1_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )

    return {
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
    }
def sample_metrics(
    y_true,
    y_pred,
    labels=(-1, 0, 1)
):
    """
    Sample-based metrics:
    - Compute metrics per sample across all aspects
    - Then average over samples
    """

    precisions, recalls, f1s = [], [], []

    for i in range(y_true.shape[0]):
        yt = y_true[i]
        yp = y_pred[i]

        precisions.append(
            precision_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        recalls.append(
            recall_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        f1s.append(
            f1_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )

    return {
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
    }


In [ ]:
test(CFG)

In [16]:
model = run(CFG)
model.eval()
model.float()
for p in model.parameters():
    assert p.dtype == torch.float32

dummy_ids = torch.ones(1, 256, dtype=torch.long).to(DEVICE)
dummy_mask = torch.ones(1, 256, dtype=torch.long).to(DEVICE)


🔹 Loading Tokenizer & Base Architecture from: Faisal191/aspect-classifier (subfolder: Domain_trained_encoder)


tokenizer_config.json: 0.00B [00:00, ?B/s]

Domain_trained_encoder/spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

Domain_trained_encoder/model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

final_stage3_v4/best_stage3.pt:   0%|          | 0.00/786M [00:00<?, ?B/s]

✅ Trained weights loaded successfully!
   Unexpected keys: 3 ['prob_weight', 'sent_norm3.weight', 'sent_norm3.bias']


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
torch.onnx.export(
    model,
    (dummy_ids, dummy_mask),
    "Habsa_v4_fp32.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["top_logits", "sub_logits", "sent_logits"],
    dynamic_axes={
        "input_ids": {0: "batch"},
        "attention_mask": {0: "batch"},
        "top_logits": {0: "batch"},
        "sub_logits": {0: "batch"},
        "sent_logits": {0: "batch"},
    },
    opset_version=17,
    do_constant_folding=False,
    dynamo=False)


/tmp/ipython-input-1862998569.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/tmp/ipython-input-3162154572.py:64: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  HALF = torch.tensor(0.

In [ ]:
hf_token = userdata.get("HF_TOKEN")
api = HfApi(token=hf_token)

repo_id = "Faisal191/aspect-classifier"

api.upload_file(
    path_or_fileobj=Path("/content/Habsa_v4_fp32.onnx"),
    path_in_repo="HABSA/Habsa_v4_fp32.onnx",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Add FP16 ONNX graph"
)

print(f"✅ FP16 ONNX model uploaded to https://huggingface.co/{repo_id}/tree/main/HABSA")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/Habsa_v4_fp32.onnx :   5%|5         | 42.1MB /  786MB            

✅ FP16 ONNX model uploaded to https://huggingface.co/Faisal191/aspect-classifier/tree/main/HABSA
